# 평가 · 체크포인트 및 테스트 데이터

In [ ]:
import os
import sys
from pathlib import Path

project = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "modules/vocabulary.py").is_file())
os.chdir(project)
if str(project) not in sys.path:
    sys.path.insert(0, str(project))
os.environ["SIGNBRIDGE_VOCABULARY"] = "assistant_demo"
from modules.vocabulary import NUM_CLASSES
if NUM_CLASSES != 13:
    raise RuntimeError("Restart the kernel to load the assistant_demo vocabulary.")
import json
import numpy as np
import torch
from torch.utils.data import DataLoader

from modules.data import KSLDataset, read_manifest
from modules.models import SignLanguageModel

checkpoint_path = Path("models/assistant_demo_325_loso/final.pth")
manifest = Path("data/processed_assistant_demo_325/manifest.csv")
output_path = Path("reports/assistant_demo_325_loso/evaluation.json")

## 테스트 집합별 정확도, 혼동행렬, 단어별 결과

In [ ]:
def macro_f1(labels, predictions, num_classes):
    scores = []
    for class_index in range(num_classes):
        true_positive = np.sum((labels == class_index) & (predictions == class_index))
        false_positive = np.sum((labels != class_index) & (predictions == class_index))
        false_negative = np.sum((labels == class_index) & (predictions != class_index))
        denominator = 2 * true_positive + false_positive + false_negative
        scores.append(0.0 if denominator == 0 else 2 * true_positive / denominator)
    return float(np.mean(scores))


def evaluate(model, samples, device, batch_size):
    if not samples:
        return None
    loader = DataLoader(KSLDataset(samples), batch_size=batch_size, shuffle=False)
    labels, top5 = [], []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            logits = model(batch["x"].to(device))
            k = min(5, logits.shape[1])
            top5.append(logits.topk(k, dim=1).indices.cpu().numpy())
            labels.append(batch["y"].numpy())
    labels = np.concatenate(labels)
    top5 = np.concatenate(top5)
    predictions = top5[:, 0]
    result = {
        "samples": int(len(labels)),
        "top1_accuracy": float(np.mean(predictions == labels)),
        "top3_accuracy": float(np.mean([label in row[:3] for label, row in zip(labels, top5)])),
        "top5_accuracy": float(np.mean([label in row for label, row in zip(labels, top5)])),
        "macro_f1": macro_f1(labels, predictions, model.config["num_classes"]),
    }
    confusion = np.zeros((model.config["num_classes"], model.config["num_classes"]), dtype=int)
    np.add.at(confusion, (labels, predictions), 1)
    result["confusion_matrix"] = confusion.tolist()
    result["per_class"] = {
        str(label): {"samples": int(np.sum(labels == label)),
                     "correct": int(np.sum((labels == label) & (predictions == label)))}
        for label in sorted(set(labels.tolist()))
    }
    result["predictions"] = [
        {"source_sequence": str(sample.path), "true_label": int(label),
         "predicted_label": int(prediction), "top3": row[:3].tolist()}
        for sample, label, prediction, row in zip(samples, labels, predictions, top5)
    ]
    return result

## 모델 로드 및 평가 결과 저장

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(checkpoint_path, map_location=device)
if checkpoint.get("format_version") != 2:
    raise ValueError("This evaluator requires a format_version=2 checkpoint")
model = SignLanguageModel(**checkpoint["model_config"]).to(device)
model.load_state_dict(checkpoint["model_state"])

held_out_signer = checkpoint.get("held_out_signer")
samples = read_manifest(manifest)
checkpoint_map = checkpoint["label_map"]
if any(checkpoint_map.get(sample.class_key) != sample.label for sample in samples):
    raise ValueError("Manifest labels do not match this checkpoint")
held_out = [
    sample
    for sample in samples
    if sample.domain == "team" and sample.signer_id == held_out_signer
]
external = [sample for sample in samples if sample.domain == "external"]
report = {
    "checkpoint": str(checkpoint_path.resolve()),
    "word_dictionary": checkpoint.get("word_dictionary", {}),
    "label_map": checkpoint.get("label_map", {}),
    "held_out_signer": held_out_signer,
    "held_out_team": evaluate(model, held_out, device, 64),
    "external_fixed_test": evaluate(model, external, device, 64),
}

output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({key: ({k: v for k, v in value.items() if k != "predictions"}
                       if key in {"held_out_team", "external_fixed_test"} and value else value)
                  for key, value in report.items()}, ensure_ascii=False, indent=2))

## Top-1·3·5 정확도 비교

In [ ]:
import matplotlib.pyplot as plt

names, top1, top3, top5 = [], [], [], []
for name in ("held_out_team", "external_fixed_test"):
    metrics = report.get(name)
    if metrics:
        names.append(name)
        top1.append(metrics["top1_accuracy"] * 100)
        top3.append(metrics["top3_accuracy"] * 100)
        top5.append(metrics["top5_accuracy"] * 100)

x = range(len(names))
plt.bar([value - 0.25 for value in x], top1, width=0.25, label="Top-1")
plt.bar(x, top3, width=0.25, label="Top-3")
plt.bar([value + 0.25 for value in x], top5, width=0.25, label="Top-5")
plt.xticks(list(x), names)
plt.ylim(0, 100)
plt.ylabel("Accuracy (%)")
plt.legend()
plt.show()